In [1]:
import sys
print(sys.version)

3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]


In [2]:
%pip install -q pgserver psycopg2-binary sqlalchemy pandas pyarrow ucimlrepo

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 39.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.3/4.3 MB 58.8 MB/s eta 0:00:00


In [3]:
import pandas as pd
import pgserver
from sqlalchemy import create_engine, text

pg = pgserver.get_server('/tmp/hw3_pg', cleanup_mode='stop')
engine = create_engine(pg.get_uri(), future=True)

with engine.connect() as conn:
    version = conn.execute(text('SELECT version()')).scalar_one()
    print(version)

PostgreSQL 16.2 on x86_64-pc-linux-gnu, compiled by gcc (GCC) 10.2.1 20210130 (Red Hat 10.2.1-11), 64-bit


In [4]:
from ucimlrepo import fetch_ucirepo

# Завантажуємо Bank Marketing dataset
dataset = fetch_ucirepo(id=222)

# Об'єднуємо features і target
df = pd.concat(
    [dataset.data.features, dataset.data.targets],
    axis=1
)

# Зберігаємо у CSV
csv_path = '/content/hw3_bank_marketing.csv'
df.to_csv(csv_path, index=False)

print("Rows and columns:", df.shape)
display(df.head())

Rows and columns: (45211, 17)


,age,job,marital,education,default,balance,housing,loan,contact,day_of_week,month,duration,campaign,pdays,previous,poutcome,y
0,58,management,married,tertiary,no,2143,yes,no,NaN,5,may,261,1,-1,0,NaN,no
1,44,technician,single,secondary,no,29,yes,no,NaN,5,may,151,1,-1,0,NaN,no
2,33,entrepreneur,married,secondary,no,2,yes,yes,NaN,5,may,76,1,-1,0,NaN,no
3,47,blue-collar,married,NaN,no,1506,yes,no,NaN,5,may,92,1,-1,0,NaN,no
4,33,NaN,single,NaN,no,1,no,no,NaN,5,may,198,1,-1,0,NaN,no


In [5]:
print(df.columns.tolist())

['age', 'job', 'marital', 'education', 'default', 'balance', 'housing', 'loan', 'contact', 'day_of_week', 'month', 'duration', 'campaign', 'pdays', 'previous', 'poutcome', 'y']


In [6]:
print(df.columns.tolist())

['age', 'job', 'marital', 'education', 'default', 'balance', 'housing', 'loan', 'contact', 'day_of_week', 'month', 'duration', 'campaign', 'pdays', 'previous', 'poutcome', 'y']


In [7]:
from sqlalchemy import text

with engine.begin() as conn:
    conn.execute(text("""
        DROP TABLE IF EXISTS hw3_bank_clean CASCADE;
        DROP TABLE IF EXISTS hw3_bank_staging CASCADE;

        CREATE TABLE hw3_bank_staging (
            age_raw TEXT,
            job_raw TEXT,
            marital_raw TEXT,
            education_raw TEXT,
            default_raw TEXT,
            balance_raw TEXT,
            housing_raw TEXT,
            loan_raw TEXT,
            contact_raw TEXT,
            day_of_week_raw TEXT,
            month_raw TEXT,
            duration_raw TEXT,
            campaign_raw TEXT,
            pdays_raw TEXT,
            previous_raw TEXT,
            poutcome_raw TEXT,
            y_raw TEXT
        );
    """))

print("Staging table created!")

Staging table created!


In [8]:
raw_conn = engine.raw_connection()

try:
    with raw_conn.cursor() as cur:
        with open(csv_path, 'r', encoding='utf-8', newline='') as file:
            cur.copy_expert(
                """
                COPY hw3_bank_staging (
                    age_raw,
                    job_raw,
                    marital_raw,
                    education_raw,
                    default_raw,
                    balance_raw,
                    housing_raw,
                    loan_raw,
                    contact_raw,
                    day_of_week_raw,
                    month_raw,
                    duration_raw,
                    campaign_raw,
                    pdays_raw,
                    previous_raw,
                    poutcome_raw,
                    y_raw
                )
                FROM STDIN
                WITH (
                    FORMAT csv,
                    HEADER true,
                    DELIMITER ','
                )
                """,
                file
            )

    raw_conn.commit()

finally:
    raw_conn.close()

print("CSV loaded into staging!")

CSV loaded into staging!


In [9]:
pd.read_sql(
    "SELECT COUNT(*) AS loaded_rows FROM hw3_bank_staging",
    engine
)

,loaded_rows
0,45211


In [10]:
with engine.begin() as conn:
    conn.execute(text("""
        DROP TABLE IF EXISTS hw3_bank_clean CASCADE;

        CREATE TABLE hw3_bank_clean (
            row_id BIGINT
                GENERATED BY DEFAULT AS IDENTITY
                PRIMARY KEY,

            age INTEGER NOT NULL
                CHECK (age BETWEEN 18 AND 100),

            job TEXT,
            marital TEXT,
            education TEXT,

            has_default BOOLEAN,
            balance INTEGER,

            has_housing BOOLEAN,
            has_loan BOOLEAN,

            contact TEXT,
            day_of_week INTEGER,
            month TEXT,

            duration INTEGER
                CHECK (duration >= 0),

            campaign INTEGER
                CHECK (campaign >= 0),

            pdays INTEGER,
            previous INTEGER
                CHECK (previous >= 0),

            poutcome TEXT,

            subscribed BOOLEAN NOT NULL
        );
    """))

print("Clean table created!")

Clean table created!


In [11]:
with engine.begin() as conn:
    conn.execute(text("""
        INSERT INTO hw3_bank_clean (
            age,
            job,
            marital,
            education,
            has_default,
            balance,
            has_housing,
            has_loan,
            contact,
            day_of_week,
            month,
            duration,
            campaign,
            pdays,
            previous,
            poutcome,
            subscribed
        )

        SELECT
            NULLIF(TRIM(age_raw), '')::INTEGER,

            NULLIF(TRIM(job_raw), ''),
            NULLIF(TRIM(marital_raw), ''),
            NULLIF(TRIM(education_raw), ''),

            CASE
                WHEN LOWER(TRIM(default_raw)) = 'yes' THEN TRUE
                WHEN LOWER(TRIM(default_raw)) = 'no' THEN FALSE
                ELSE NULL
            END,

            NULLIF(TRIM(balance_raw), '')::INTEGER,

            CASE
                WHEN LOWER(TRIM(housing_raw)) = 'yes' THEN TRUE
                WHEN LOWER(TRIM(housing_raw)) = 'no' THEN FALSE
                ELSE NULL
            END,

            CASE
                WHEN LOWER(TRIM(loan_raw)) = 'yes' THEN TRUE
                WHEN LOWER(TRIM(loan_raw)) = 'no' THEN FALSE
                ELSE NULL
            END,

            NULLIF(TRIM(contact_raw), ''),
            NULLIF(TRIM(day_of_week_raw), '')::INTEGER,
            NULLIF(TRIM(month_raw), ''),

            NULLIF(TRIM(duration_raw), '')::INTEGER,
            NULLIF(TRIM(campaign_raw), '')::INTEGER,
            NULLIF(TRIM(pdays_raw), '')::INTEGER,
            NULLIF(TRIM(previous_raw), '')::INTEGER,

            NULLIF(TRIM(poutcome_raw), ''),

            CASE
                WHEN LOWER(TRIM(y_raw)) = 'yes' THEN TRUE
                WHEN LOWER(TRIM(y_raw)) = 'no' THEN FALSE
            END

        FROM hw3_bank_staging

        WHERE NULLIF(TRIM(age_raw), '') IS NOT NULL
          AND NULLIF(TRIM(y_raw), '') IS NOT NULL;
    """))

print("Clean data loaded!")

Clean data loaded!


In [12]:
pd.read_sql("""
    SELECT *
    FROM hw3_bank_clean
    LIMIT 5
""", engine)

,row_id,age,job,marital,education,has_default,balance,has_housing,has_loan,contact,day_of_week,month,duration,campaign,pdays,previous,poutcome,subscribed
0,1,58,management,married,tertiary,False,2143,True,False,None,5,may,261,1,-1,0,None,False
1,2,44,technician,single,secondary,False,29,True,False,None,5,may,151,1,-1,0,None,False
2,3,33,entrepreneur,married,secondary,False,2,True,True,None,5,may,76,1,-1,0,None,False
3,4,47,blue-collar,married,None,False,1506,True,False,None,5,may,92,1,-1,0,None,False
4,5,33,None,single,None,False,1,False,False,None,5,may,198,1,-1,0,None,False


In [13]:
pd.read_sql("""
    SELECT COUNT(*) AS clean_rows
    FROM hw3_bank_clean
""", engine)

,clean_rows
0,45211


## Завдання 2. Staging і DDL

У staging-таблиці всі сирі значення зберігаються як `TEXT`. Це дозволяє завантажити дані без помилок навіть тоді, коли у файлі є порожні значення або спеціальні категорії.

У фінальній таблиці `hw3_bank_clean` я використала правильні типи даних:

- `age`, `balance`, `duration`, `campaign`, `pdays`, `previous` - `INTEGER`, тому що це числові значення;
- `job`, `marital`, `education`, `contact`, `month`, `poutcome` -`TEXT`, тому що це категоріальні значення без фіксованого бізнес-ліміту;
- `has_default`, `has_housing`, `has_loan`, `subscribed` `BOOLEAN`, тому що початкові значення `yes/no` були перетворені на `TRUE/FALSE`;
- `row_id` технічний `PRIMARY KEY`.

Для очищення використовувалися `TRIM` та `NULLIF`, щоб прибрати зайві пробіли та перетворити порожні рядки на `NULL`.

Для логічних колонок використано `CASE`, щоб нормалізувати значення `yes/no`.

Також додано `CHECK`-обмеження. Наприклад, вік повинен бути від 18 до 100 років, а `duration`, `campaign` і `previous` не можуть бути від’ємними.

In [14]:
row_counts = pd.read_sql("""
    SELECT 'staging' AS layer, COUNT(*) AS n_rows
    FROM hw3_bank_staging

    UNION ALL

    SELECT 'clean' AS layer, COUNT(*) AS n_rows
    FROM hw3_bank_clean;
""", engine)

row_counts

,layer,n_rows
0,staging,45211
1,clean,45211


### Висновок щодо кількості рядків

У staging-таблиці є 45 211 рядків, і у clean-таблиці також 45 211 рядків.

Кількість однакова, тому що під час очищення не було відфільтровано жодного запису. Усі рядки успішно пройшли перетворення типів і були перенесені у фінальну таблицю.

In [15]:
duplicates = pd.read_sql("""
    SELECT
        COUNT(*) AS total_rows,

        COUNT(
            DISTINCT (
                age,
                job,
                marital,
                education,
                balance,
                duration,
                campaign,
                subscribed
            )
        ) AS distinct_business_keys,

        COUNT(*) -
        COUNT(
            DISTINCT (
                age,
                job,
                marital,
                education,
                balance,
                duration,
                campaign,
                subscribed
            )
        ) AS duplicate_rows

    FROM hw3_bank_clean;
""", engine)

duplicates

,total_rows,distinct_business_keys,duplicate_rows
0,45211,45199,12


In [16]:
null_audit = pd.read_sql("""
    SELECT
        ROUND(
            COUNT(*) FILTER (WHERE job IS NULL) * 100.0
            / NULLIF(COUNT(*), 0),
            2
        ) AS pct_null_job,

        ROUND(
            COUNT(*) FILTER (WHERE education IS NULL) * 100.0
            / NULLIF(COUNT(*), 0),
            2
        ) AS pct_null_education,

        ROUND(
            COUNT(*) FILTER (WHERE contact IS NULL) * 100.0
            / NULLIF(COUNT(*), 0),
            2
        ) AS pct_null_contact,

        ROUND(
            COUNT(*) FILTER (WHERE poutcome IS NULL) * 100.0
            / NULLIF(COUNT(*), 0),
            2
        ) AS pct_null_poutcome,

        ROUND(
            COUNT(*) FILTER (WHERE balance IS NULL) * 100.0
            / NULLIF(COUNT(*), 0),
            2
        ) AS pct_null_balance,

        COUNT(*) AS total_rows

    FROM hw3_bank_clean;
""", engine)

null_audit

,pct_null_job,pct_null_education,pct_null_contact,pct_null_poutcome,pct_null_balance,total_rows
0,0.64,4.11,28.8,81.75,0.0,45211


In [17]:
special_markers = pd.read_sql("""
    SELECT
        COUNT(*) FILTER (
            WHERE job_raw IS NULL
               OR TRIM(job_raw) = ''
               OR LOWER(TRIM(job_raw)) = 'unknown'
               OR TRIM(job_raw) = '?'
        ) AS missing_or_unknown_job,

        COUNT(*) FILTER (
            WHERE education_raw IS NULL
               OR TRIM(education_raw) = ''
               OR LOWER(TRIM(education_raw)) = 'unknown'
               OR TRIM(education_raw) = '?'
        ) AS missing_or_unknown_education,

        COUNT(*) FILTER (
            WHERE contact_raw IS NULL
               OR TRIM(contact_raw) = ''
               OR LOWER(TRIM(contact_raw)) = 'unknown'
               OR TRIM(contact_raw) = '?'
        ) AS missing_or_unknown_contact,

        COUNT(*) FILTER (
            WHERE poutcome_raw IS NULL
               OR TRIM(poutcome_raw) = ''
               OR LOWER(TRIM(poutcome_raw)) = 'unknown'
               OR TRIM(poutcome_raw) = '?'
        ) AS missing_or_unknown_poutcome,

        COUNT(*) AS total_rows

    FROM hw3_bank_staging;
""", engine)

special_markers

,missing_or_unknown_job,missing_or_unknown_education,missing_or_unknown_contact,missing_or_unknown_poutcome,total_rows
0,288,1857,13020,36959,45211


### Висновок щодо пропусків і спеціальних значень

У сирих даних були знайдені спеціальні або відсутні значення.

- `job`: 288 записів;
- `education`: 1 857 записів;
- `contact`: 13 020 записів;
- `poutcome`: 36 959 записів.

Найбільше таких значень у колонці `poutcome`. Це приблизно 82% всього датасету. Це можна пояснити тим, що для багатьох клієнтів результат попередньої маркетингової кампанії був невідомий або попереднього контакту не було.

Колонка `contact` також має досить багато невідомих значень. Такі категорії не варто автоматично видаляти, тому що вони можуть містити корисну інформацію. Під час ML-підготовки їх можна залишити як окрему категорію або обробити окремим правилом.

In [18]:
subscription_distribution = pd.read_sql("""
    SELECT
        subscribed,
        COUNT(*) AS n_rows,
        ROUND(
            COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (),
            2
        ) AS pct_rows
    FROM hw3_bank_clean
    GROUP BY subscribed
    ORDER BY n_rows DESC;
""", engine)

subscription_distribution

,subscribed,n_rows,pct_rows
0,False,39922,88.3
1,True,5289,11.7


### Висновок щодо target `subscribed`

У датасеті більшість клієнтів не підписалися на депозит.

- `False`: 39 922 записів (88.3%);
- `True`: 5 289 записів (11.7%).

Отже, цільова змінна є незбалансованою. Це важливо враховувати під час ML, тому що модель може частіше передбачати клас `False` просто через його більшу кількість.

In [19]:
balance_stats = pd.read_sql("""
    SELECT
        COUNT(balance) AS n_non_null,
        MIN(balance) AS min_value,
        MAX(balance) AS max_value,
        ROUND(AVG(balance)::NUMERIC, 2) AS avg_value,
        ROUND(STDDEV_SAMP(balance)::NUMERIC, 2) AS sd_value
    FROM hw3_bank_clean;
""", engine)

balance_stats

,n_non_null,min_value,max_value,avg_value,sd_value
0,45211,-8019,102127,1362.27,3044.77


### Висновок щодо `balance`

Колонка `balance` не має пропусків.

Мінімальне значення становить -8019, максимальне — 102127, а середнє значення — 1362.27.

Розкид значень досить великий, оскільки стандартне відхилення становить 3044.77. Це означає, що у вибірці є клієнти з дуже великими або дуже низькими балансами.

Від’ємні значення не обов’язково є помилкою, тому що вони можуть означати борг або від’ємний залишок на рахунку. Великі значення також потрібно перевіряти, але не видаляти автоматично.

In [20]:
outliers = pd.read_sql("""
    WITH stats AS (
        SELECT
            AVG(balance) AS mean_value,
            STDDEV_SAMP(balance) AS sd_value
        FROM hw3_bank_clean
        WHERE balance IS NOT NULL
    )

    SELECT
        row_id,
        age,
        job,
        balance,
        subscribed,

        ROUND(
            ((balance - mean_value) / NULLIF(sd_value, 0))::NUMERIC,
            2
        ) AS z_score

    FROM hw3_bank_clean
    CROSS JOIN stats

    WHERE balance IS NOT NULL
      AND sd_value > 0
      AND ABS(balance - mean_value) > 3 * sd_value

    ORDER BY ABS(balance - mean_value) DESC
    LIMIT 20;
""", engine)

outliers

,row_id,age,job,balance,subscribed,z_score
0,39990,51,management,102127,False,33.09
1,26228,59,management,98417,False,31.88
2,43394,84,retired,81204,True,26.22
3,42559,84,retired,81204,True,26.22
4,41694,60,retired,71188,False,22.93
5,19786,56,management,66721,False,21.47
6,21193,52,blue-collar,66653,False,21.44
7,19421,59,admin.,64343,False,20.68
8,41375,32,entrepreneur,59649,False,19.14
9,12927,56,blue-collar,58932,False,18.91


### Висновок щодо outliers

Перевірка за правилом 3σ знайшла клієнтів із дуже великими значеннями `balance`.

Найбільше значення становить 102127 і має z-score 33.09. Також є інші великі значення з високими z-score.

Ці записи є потенційними outliers, але я не видаляю їх автоматично. Великий баланс може бути реальним значенням для окремого клієнта.

Для реального проєкту я б додатково перевірила такі записи у джерелі даних або створила окремий прапорець `is_balance_outlier`.

### Query 1

**Business-question:** Які типи роботи найчастіше зустрічаються серед клієнтів?

In [21]:
query_1 = pd.read_sql("""
    SELECT
        job,
        COUNT(*) AS n_clients
    FROM hw3_bank_clean
    WHERE job IS NOT NULL
    GROUP BY job
    ORDER BY n_clients DESC, job
    LIMIT 10;
""", engine)

query_1

,job,n_clients
0,blue-collar,9732
1,management,9458
2,technician,7597
3,admin.,5171
4,services,4154
5,retired,2264
6,self-employed,1579
7,entrepreneur,1487
8,unemployed,1303
9,housemaid,1240


**Interpretation:**  
Найбільше клієнтів у датасеті мають роботу `blue-collar` — 9732 записи.  
Друга найбільша група — `management` з 9458 записами, а третя — `technician` з 7597.

Отже, ці три категорії найбільше представлені в наборі даних. Це важливо враховувати під час аналізу, тому що деякі професії представлені значно частіше за інші.

### Query 2

**Business-question:** Який відсоток клієнтів підписався на депозит у різних типах роботи?

In [22]:
query_2 = pd.read_sql("""
    SELECT
        job,
        COUNT(*) AS total_clients,
        COUNT(*) FILTER (WHERE subscribed = TRUE) AS subscribed_clients,
        ROUND(
            COUNT(*) FILTER (WHERE subscribed = TRUE) * 100.0
            / NULLIF(COUNT(*), 0),
            2
        ) AS subscription_rate
    FROM hw3_bank_clean
    WHERE job IS NOT NULL
    GROUP BY job
    ORDER BY subscription_rate DESC, job
    LIMIT 10;
""", engine)

query_2

,job,total_clients,subscribed_clients,subscription_rate
0,student,938,269,28.68
1,retired,2264,516,22.79
2,unemployed,1303,202,15.50
3,management,9458,1301,13.76
4,admin.,5171,631,12.20
5,self-employed,1579,187,11.84
6,technician,7597,840,11.06
7,services,4154,369,8.88
8,housemaid,1240,109,8.79
9,entrepreneur,1487,123,8.27


**Interpretation:**  
Найвищий відсоток підписки мають студенти — 28.68%.  
На другому місці клієнти зі статусом `retired` — 22.79%, а далі `unemployed` — 15.50%.

Цікаво, що `management` має багато клієнтів, але рівень підписки там нижчий — 13.76%. Отже, велика кількість клієнтів у категорії не означає високий відсоток успішної підписки.

### Query 3

**Business-question:** Які клієнти віком від 30 до 50 років мають великий баланс і при цьому не мають хоча б одного типу кредитного навантаження?

In [23]:
query_3 = pd.read_sql("""
    SELECT
        row_id,
        age,
        job,
        balance,
        has_housing,
        has_loan,
        subscribed
    FROM hw3_bank_clean
    WHERE age BETWEEN 30 AND 50
      AND balance > 5000
      AND (
            has_housing = FALSE
            OR has_loan = FALSE
          )
      AND NOT job = 'unknown'
    ORDER BY balance DESC
    LIMIT 10;
""", engine)

query_3

,row_id,age,job,balance,has_housing,has_loan,subscribed
0,41375,32,entrepreneur,59649,False,False,False
1,922,44,admin.,58544,True,False,False
2,27391,50,services,57435,True,False,False
3,3198,43,admin.,56831,False,False,False
4,22092,40,management,52527,False,False,False
5,15870,38,admin.,45789,False,False,False
6,448,39,technician,45248,True,False,True
7,27050,31,management,44128,False,False,False
8,19796,42,entrepreneur,42045,False,False,False
9,21347,43,management,41630,False,False,False


**Interpretation:**  
У топ-10 клієнтів за балансом у цьому запиті значення балансу дуже високі — приблизно від 41 630 до 59 649.

Більшість цих клієнтів не підписались на депозит. Лише один клієнт у показаних результатах має `subscribed = TRUE`.

Це показує, що великий баланс сам по собі не гарантує підписку на депозит.

### Query 4

**Business-question:** Які категорії освіти, що закінчуються на `ary`, найчастіше зустрічаються в датасеті?

In [25]:
query_4 = pd.read_sql(
    text("""
        SELECT
            education,
            COUNT(*) AS n_clients
        FROM hw3_bank_clean
        WHERE education ILIKE :pattern
        GROUP BY education
        ORDER BY n_clients DESC;
    """),
    engine,
    params={"pattern": "%ary"}
)

query_4

,education,n_clients
0,secondary,23202
1,tertiary,13301
2,primary,6851


In [26]:
params={"pattern": "%ary"}

### Query 5

**Business-question:** Які унікальні значення `contact` є в датасеті?

In [27]:
query_5 = pd.read_sql("""
    SELECT DISTINCT
        contact
    FROM hw3_bank_clean
    WHERE contact IS NOT NULL
    ORDER BY contact;
""", engine)

query_5

,contact
0,cellular
1,telephone


**Interpretation:**  
У clean-таблиці є два відомі типи контакту: `cellular` і `telephone`.

Це означає, що після очищення даних основні канали контакту представлені двома категоріями. Значення `unknown`, які були в сирих даних, не потрапили в цей список, тому що в запиті використано `IS NOT NULL`.

### Query 6

**Business-question:** Які значення `poutcome` є в даних, якщо пропущені значення показати як `no_previous_result`?

In [28]:
query_6 = pd.read_sql("""
    SELECT
        COALESCE(poutcome, 'no_previous_result') AS previous_outcome,
        COUNT(*) AS n_clients
    FROM hw3_bank_clean
    GROUP BY COALESCE(poutcome, 'no_previous_result')
    ORDER BY n_clients DESC;
""", engine)

query_6

,previous_outcome,n_clients
0,no_previous_result,36959
1,failure,4901
2,other,1840
3,success,1511


**Interpretation:**  
У більшості клієнтів немає відомого результату попередньої маркетингової кампанії — 36 959 записів.

Серед відомих результатів найчастіше зустрічається `failure` — 4 901 запис.  
Категорія `success` зустрічається значно рідше — 1 511 записів.

Це означає, що `poutcome` має багато пропущеної інформації, тому цю ознаку потрібно використовувати обережно в аналізі та ML.

### Query 7

**Business-question:** Скільки клієнтів працюють у трьох вибраних професіях: `management`, `technician` або `admin.`?

In [29]:
query_7 = pd.read_sql("""
    SELECT
        job,
        COUNT(*) AS n_clients
    FROM hw3_bank_clean
    WHERE job IN ('management', 'technician', 'admin.')
    GROUP BY job
    ORDER BY n_clients DESC;
""", engine)

query_7

,job,n_clients
0,management,9458
1,technician,7597
2,admin.,5171


**Interpretation:**  
Серед трьох вибраних професій найбільше клієнтів працюють у `management` — 9458.

Далі йдуть `technician` — 7597 та `admin.` — 5171.

Отже, серед цих трьох категорій `management` представлена найбільше.

### Query 8

**Business-question:** Скільки клієнтів мають різний статус housing loan і personal loan?

In [30]:
query_8 = pd.read_sql("""
    SELECT
        COUNT(*) AS different_loan_status
    FROM hw3_bank_clean
    WHERE has_housing IS DISTINCT FROM has_loan;
""", engine)

query_8

,different_loan_status
0,23640


**Interpretation:**  
Запит показує кількість клієнтів, у яких статус житлового кредиту і персонального кредиту відрізняється.

`IS DISTINCT FROM` зручно використовувати для такого порівняння, тому що воно коректно працює навіть із NULL.

### Query 9

**Business-question:** У які місяці банк найчастіше контактував із клієнтами?

In [31]:
query_9 = pd.read_sql("""
    SELECT
        month,
        COUNT(*) AS n_contacts
    FROM hw3_bank_clean
    WHERE month IS NOT NULL
    GROUP BY month
    ORDER BY n_contacts DESC
    LIMIT 5;
""", engine)

query_9

,month,n_contacts
0,may,13766
1,jul,6895
2,aug,6247
3,jun,5341
4,nov,3970


**Interpretation:**  
Результат показує п’ять місяців із найбільшою кількістю контактів.

Найчастіше банк контактував із клієнтами в місяці, який знаходиться на першому місці результату. Це показує, що активність маркетингової кампанії була нерівномірною протягом року.

### Query 10

**Business-question:** Як відрізняється середня тривалість дзвінка для клієнтів, які підписались і не підписались?

In [32]:
query_10 = pd.read_sql("""
    SELECT
        subscribed,
        COUNT(*) AS n_clients,
        ROUND(AVG(duration)::NUMERIC, 2) AS avg_duration
    FROM hw3_bank_clean
    WHERE duration IS NOT NULL
    GROUP BY subscribed
    ORDER BY subscribed;
""", engine)

query_10

,subscribed,n_clients,avg_duration
0,False,39922,221.18
1,True,5289,537.29


**Interpretation:**  
Середня тривалість дзвінка відрізняється між клієнтами, які підписались, і тими, які не підписались.

Це може показувати сильний зв’язок між `duration` і результатом кампанії. Але `duration` потрібно використовувати обережно для ML, тому що тривалість дзвінка стає відомою лише після завершення контакту і може створити data leakage.

## Завдання 6. Reflection

Під час роботи з датасетом Bank Marketing я побачила кілька проблем якості даних. У деяких колонках були спеціальні значення `unknown`, особливо в `contact` і `poutcome`. Також у даних є потенційні дублікати та дуже великі значення `balance`, які можна вважати outliers. Водночас великі значення не обов’язково є помилкою, тому їх не варто видаляти автоматично.

Для ML я б використала такі ознаки, як `age`, `job`, `marital`, `education`, `balance`, `housing`, `loan`, `contact`, `campaign`, `previous` і `poutcome`. Вони можуть бути корисними для прогнозування підписки клієнта.

Колонку `duration` я б не використовувала як feature, якщо прогноз робиться до або на початку дзвінка. Тривалість дзвінка стає відомою тільки після його завершення, тому вона може створити data leakage.

Для покращення pipeline я б запропонувала автоматично перевіряти типи даних, спеціальні значення, кількість рядків, дублікати та аномальні значення перед завантаженням у фінальну таблицю. Також корисно зберігати окремо raw/staging і clean-рівні.

Для цього датасету природною ML-задачею є classification, тому що цільова змінна `subscribed` має два значення: клієнт підписався або не підписався.